# GPU benchmark: PyTorch ticket classifier on CPU vs NVIDIA GPU

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

The notebook trains the model from `gpu/textclf.py`, runs the unit tests (including the CUDA-vs-CPU consistency test) and benchmarks CPU and GPU with warmup and `torch.cuda.synchronize()`. The results JSON is downloaded at the end.

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess
REPO = "https://github.com/AlonaTsk/ai-inference-platform.git"
if not os.path.exists("ai-inference-platform"):
    if subprocess.run(["git", "clone", "--depth", "1", REPO]).returncode != 0:
        # Repo not public (yet): upload dist/gpu-bench.zip built with `make gpu-zip`.
        from google.colab import files
        print("Clone failed. Upload gpu-bench.zip:")
        name = next(iter(files.upload()))
        subprocess.run(["unzip", "-q", name, "-d", "ai-inference-platform"], check=True)
%cd ai-inference-platform

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: set Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))

In [ ]:
!cd gpu && python -m pytest -q -rs

In [ ]:
!cd gpu && python benchmark.py --require-cuda --fp16 --json results/colab-gpu.json

In [ ]:
from google.colab import files
files.download("gpu/results/colab-gpu.json")